## 1. Basic Imports & Environment Setup
This section imports essential Python libraries for data manipulation (Pandas, NumPy), text visualization (WordCloud, Matplotlib), and Natural Language Processing (NLTK).


In [3]:
# Import NumPy library for array operations, linear algebra, and numerical matrix processing
import numpy as np        # np alias for NumPy multi-dimensional array manipulation

# Import Pandas library for data loading, data wrangling, and tabular DataFrame management
import pandas as pd       # pd alias for Pandas DataFrame structures

# Import Matplotlib pyplot module for rendering static, interactive, and animated data plots
import matplotlib.pyplot as plt  # plt alias for plotting figures and charts

# Jupyter magic command to render matplotlib graphics directly below code cells
%matplotlib inline        # Enables inline plotting within Jupyter notebook interface

# Import WordCloud object to visualize word frequencies as word cloud images
from wordcloud import WordCloud  # WordCloud generator class

# Import Natural Language Toolkit (NLTK) for natural language processing tasks
import nltk               # NLTK framework for text tokenization, corpora, and stemming

# Import English stopwords list from NLTK corpus
from nltk.corpus import stopwords    # Stopwords dictionary containing common uninformative words

# Download the NLTK 'stopwords' dataset to local cache directory
nltk.download('stopwords')   # Downloads English stop words corpus if not present

# Download the NLTK 'punkt' tokenizer model dataset to local cache directory
nltk.download('punkt')       # Downloads Punkt sentence boundary and word tokenizer model


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Personal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Personal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
# Read the target SMS dataset CSV file into a Pandas DataFrame
df = pd.read_csv('spam.csv')  # Loads 'spam.csv' dataset file into DataFrame object 'df'

# Display the top 5 initial rows of the DataFrame to inspect data format and columns
df.head()  # Returns first 5 records of loaded DataFrame


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
# Drop unneeded unnamed empty columns ('Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4') from DataFrame
df.drop(columns = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace = True)  # Modifies df directly in-place without returning copy

# Display top 5 rows of DataFrame to confirm removal of unnamed columns
df.head()  # Returns first 5 records after column deletion


,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# Rename dataset columns: 'v1' becomes 'target' (label) and 'v2' becomes 'text' (SMS text content)
df.rename(columns = {'v1': 'target', 'v2': 'text'}, inplace = True)  # Renames columns in-place for standardized naming

# Display top 5 rows of DataFrame to confirm new column names
df.head()  # Returns first 5 records with updated headers 'target' and 'text'


,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 2. Data Preprocessing & Encoding
In this section, text labels ('ham' / 'spam') are converted into numerical binary labels (0 / 1) using `LabelEncoder`, and duplicate entries are identified and removed.


In [7]:
# Import LabelEncoder class from scikit-learn preprocessing module
from sklearn.preprocessing import LabelEncoder  # Utility to encode target labels into numeric integers

# Create an instance of LabelEncoder object
encoder = LabelEncoder()  # Initializes LabelEncoder instance

# Fit encoder on 'target' column and transform string categories ('ham' -> 0, 'spam' -> 1)
df['target'] = encoder.fit_transform(df['target'])  # Replaces 'ham' with 0 and 'spam' with 1 in target column

# Display top 5 rows of DataFrame to verify binary numerical target encoding
df.head()  # Returns first 5 records showing numeric target labels (0 and 1)


,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
# Calculate and sum total number of duplicate rows present in the DataFrame
df.duplicated().sum()  # df.duplicated() returns boolean mask; .sum() totals True duplicate rows


403

In [9]:
# Check total record count (number of rows) in DataFrame before duplicate removal
len(df)  # Returns total row count (5572 initial rows)


5572

In [10]:
# Drop duplicate rows from DataFrame while keeping the first occurrence of each unique record
df = df.drop_duplicates(keep = 'first')  # Replaces df with deduplicated subset

# Check total record count (number of rows) in DataFrame after dropping duplicates
len(df)  # Returns updated total row count (5169 deduplicated rows)


5169

## 3. Feature Engineering & Text Normalization
This section implements the `transform_text` pipeline function which converts raw text into clean, stemmed, lowercase tokens stripped of stopwords and punctuation.


In [11]:
# Import PorterStemmer algorithm from NLTK stemming module
from nltk.stem.porter import PorterStemmer  # Reduces words to base morphological stem (e.g. 'running' -> 'run')

# Import standard string library for punctuation character sets
import string  # Gives access to string.punctuation (!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~)

# Create an instance of PorterStemmer class
ps = PorterStemmer()  # Instantiates stemmer object for word stemming operations


In [12]:
# Define custom text preprocessing function to clean, tokenize, filter, and stem input text strings
def transform_text(text):
    # Convert input text string to lowercase for uniform case insensitivity
    text = text.lower()  # Converts 'Hello World' -> 'hello world'
    
    # Tokenize lowercased text string into individual word tokens using NLTK word_tokenize
    text = nltk.word_tokenize(text)  # Splits input string into Python list of tokens
    
    # Initialize empty working list y to collect valid alphanumeric tokens
    y = []  # Empty list for filtering alphanumeric words
    
    # Iterate through each token in the tokenized text list
    for i in text:  # Loop over tokens
        # Check if current token contains only letters and digits (strips special symbols)
        if i.isalnum():  # Returns True if all characters in token are alphanumeric
            # Append valid alphanumeric token to working list y
            y.append(i)  # Retains word/number token
            
    # Shallow copy filtered alphanumeric tokens from list y back into text variable
    text = y[:]  # Copies content of y to text
    
    # Clear working list y to reuse for stopword and punctuation filtering pass
    y.clear()    # Clears list y
    
    # Iterate through each alphanumeric token in text list
    for i in text:  # Loop over alphanumeric tokens
        # Check if token is NOT in English stopwords list AND NOT in punctuation symbols
        if i not in stopwords.words('english') and i not in string.punctuation:  # Filters stop words & punctuation
            # Append meaningful content token to working list y
            y.append(i)  # Retains non-stopword token
        
    # Shallow copy filtered tokens from list y back into text variable
    text = y[:]  # Copies content of y to text
    
    # Clear working list y to reuse for final stemming pass
    y.clear()    # Clears list y
    
    # Iterate through each filtered content token in text list
    for i in text:  # Loop over filtered tokens
        # Stem token using PorterStemmer and append root stem to working list y
        y.append(ps.stem(i))  # Stems token (e.g. 'craziness' -> 'crazi')
    
    # Join stemmed tokens in list y into a single space-separated string and return
    return " ".join(y)  # Combines token list back into normalized string


In [13]:
# Execute transform_text function on a sample SMS message to test preprocessing pipeline steps
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')  # Returns transformed text string


'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [14]:
# Apply transform_text function across 'text' column rows and store output in new 'transformed_text' column
df['transformed_text'] = df['text'].apply(transform_text)  # Preprocesses all SMS messages in DataFrame

# Display top 5 rows of DataFrame to inspect original text alongside transformed text
df.head()  # Returns first 5 records with new 'transformed_text' column


,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [15]:
# Import CountVectorizer and TfidfVectorizer from scikit-learn text feature extraction module
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # Text vectorization classes

# Instantiate TfidfVectorizer configured to limit feature matrix to top 500 most frequent features
tfid = TfidfVectorizer(max_features = 500)  # Restricts TF-IDF vocabulary matrix size to 500 words


In [16]:
# Vectorize 'transformed_text' column using TF-IDF and convert sparse matrix to 2D dense NumPy array X
X = tfid.fit_transform(df['transformed_text']).toarray()  # Fits TF-IDF and creates feature matrix X of shape (5169, 500)

# Extract binary target labels (0 and 1) from DataFrame 'target' column as a 1D NumPy array y
y = df['target'].values  # Converts target column to 1D NumPy array of labels


## 4. Train-Test Dataset Splitting
Splits the dataset into 80% training data and 20% testing data for model training and evaluation.


In [17]:
# Import LogisticRegression classification model from sklearn linear_model
from sklearn.linear_model import LogisticRegression  # Linear model for binary classification

# Import Support Vector Classifier (SVC) model from sklearn svm
from sklearn.svm import SVC  # Support Vector Machine classifier

# Import Multinomial Naive Bayes model from sklearn naive_bayes
from sklearn.naive_bayes import MultinomialNB  # Probabilistic classifier ideal for text features

# Import DecisionTreeClassifier model from sklearn tree
from sklearn.tree import DecisionTreeClassifier  # Non-parametric decision tree classifier

# Import KNeighborsClassifier model from sklearn neighbors
from sklearn.neighbors import KNeighborsClassifier  # Distance-based nearest neighbors classifier

# Import RandomForestClassifier ensemble model from sklearn ensemble
from sklearn.ensemble import RandomForestClassifier  # Ensemble of bagged decision trees

# Import AdaBoostClassifier ensemble model from sklearn ensemble
from sklearn.ensemble import AdaBoostClassifier  # Adaptive Boosting ensemble algorithm

# Import BaggingClassifier ensemble model from sklearn ensemble
from sklearn.ensemble import BaggingClassifier  # Bootstrap Aggregating meta-estimator

# Import ExtraTreesClassifier ensemble model from sklearn ensemble
from sklearn.ensemble import ExtraTreesClassifier  # Extremely Randomized Trees ensemble classifier

# Import GradientBoostingClassifier ensemble model from sklearn ensemble
from sklearn.ensemble import GradientBoostingClassifier  # Gradient Boosting Decision Trees classifier

# Import XGBClassifier gradient boosting model from xgboost package
from xgboost import XGBClassifier  # High-performance optimized gradient boosting framework


## 5. Machine Learning Models Initialization
Imports and instantiates multiple classification algorithms (Linear, Tree-based, Ensemble, Naive Bayes, SVM, XGBoost) for comparative benchmark evaluation.


In [18]:
# Create dictionary mapping model string identifier names to instantiated classifier objects
clfs = {
    # Map key 'SVC' to Support Vector Classifier instance
    'SVC': svc,          # Support Vector Classifier instance
    # Map key 'KNN' to K-Nearest Neighbors Classifier instance
    'KNN': knc,          # K-Nearest Neighbors instance
    # Map key 'NB' to Multinomial Naive Bayes Classifier instance
    'NB': mnb,           # Multinomial Naive Bayes instance
    # Map key 'DT' to Decision Tree Classifier instance
    'DT': dtc,           # Decision Tree Classifier instance
    # Map key 'LR' to Logistic Regression Classifier instance
    'LR': lrc,           # Logistic Regression instance
    # Map key 'RF' to Random Forest Classifier instance
    'RF': rfc,           # Random Forest Classifier instance
    # Map key 'Adaboost' to AdaBoost Classifier instance
    'Adaboost': abc,     # AdaBoost Classifier instance
    # Map key 'Bgc' to Bagging Classifier instance
    'Bgc': bc,           # Bagging Classifier instance
    # Map key 'ETC' to Extra Trees Classifier instance
    'ETC': etc,          # Extra Trees Classifier instance
    # Map key 'GBDT' to Gradient Boosting Decision Trees instance
    'GBDT': gbdt,        # Gradient Boosting Decision Trees instance
    # Map key 'xgb' to XGBoost Classifier instance
    'xgb': xgb           # XGBoost Classifier instance
}


In [19]:
# Import accuracy_score and precision_score metrics from scikit-learn metrics package
from sklearn.metrics import accuracy_score, precision_score  # Classification performance metrics

# Define helper function train_classifier to fit model on train set and evaluate metrics on test set
def train_classifier(clfs, X_train, y_train, X_test, y_test):
    # Fit/train classifier instance using training feature matrix X_train and training label array y_train
    clfs.fit(X_train, y_train)  # Trains classifier parameters on training set
    
    # Generate predicted class labels (0 or 1) for test set feature matrix X_test
    y_pred = clfs.predict(X_test)  # Predicts target values for unseen X_test
    
    # Compute accuracy score comparing ground truth test labels y_test against predictions y_pred
    accuracy = accuracy_score(y_test, y_pred)  # Calculates accuracy ratio (correct predictions / total)
    
    # Compute precision score comparing ground truth test labels y_test against predictions y_pred
    precision = precision_score(y_test, y_pred)  # Calculates precision ratio (true positive spam / total predicted spam)
    
    # Return calculated accuracy score and precision score as a 2-tuple
    return accuracy, precision  # Returns tuple of (accuracy, precision)


In [20]:
# Initialize empty list to store calculated accuracy score of each evaluated classifier
accuracy_scores = []   # List to collect test accuracy float values

# Initialize empty list to store calculated precision score of each evaluated classifier
precision_scores = []  # List to collect test precision float values

# Loop over model names and classifier instances stored in clfs dictionary
for name, clfs in clfs.items():  # Iterates through all 11 model key-value pairs
    # Train current classifier and evaluate accuracy & precision on test set via train_classifier helper
    current_accuracy, current_precision = train_classifier(clfs, X_train, y_train, X_test, y_test)  # Returns (accuracy, precision)
    
    # Print empty line break for readable console layout output
    print()  # Prints blank line
    
    # Print current classifier identifier name
    print("For: ", name)  # Prints model name (e.g., 'SVC', 'RF', 'xgb')
    
    # Print calculated test accuracy score float value
    print("Accuracy: ", current_accuracy)  # Prints accuracy value
    
    # Print calculated test precision score float value
    print("Precision: ", current_precision)  # Prints precision value
    
    # Append current classifier accuracy score to accuracy_scores list
    accuracy_scores.append(current_accuracy)  # Appends score to list
    
    # Append current classifier precision score to precision_scores list
    precision_scores.append(current_precision)  # Appends score to list


## 6. Model Training & Evaluation Metrics
Defines evaluation function to calculate Accuracy Score and Precision Score for each classifier on the test dataset.


In [21]:
# Importing evaluation metrics functions (accuracy_score, precision_score) from sklearn metrics
from sklearn.metrics import accuracy_score, precision_score  # accuracy = correct/total, precision = true_positives/(true_positives + false_positives)

# Defining helper function to train a model and compute test set accuracy and precision
def train_classifier(clfs, X_train, y_train, X_test, y_test):
    # Fit the classifier using training feature matrix X_train and training labels y_train
    clfs.fit(X_train, y_train)  # Fits/trains model parameters on training dataset
    
    # Generate predictions on unseen test dataset feature matrix X_test
    y_pred = clfs.predict(X_test)  # Predicts target class values (0 or 1) for X_test
    
    # Calculate accuracy score comparing true test labels y_test against predictions y_pred
    accuracy = accuracy_score(y_test, y_pred)  # Computes overall classification accuracy
    
    # Calculate precision score comparing true test labels y_test against predictions y_pred
    precision = precision_score(y_test, y_pred)  # Computes precision ratio (vital for minimizing false positive spam classifications)
    
    # Return calculated accuracy score and precision score tuple
    return accuracy, precision


In [22]:
# Initializing empty lists to store evaluation metrics results for each model iteration
accuracy_scores = []   # List to collect calculated accuracy score for each classifier
precision_scores = []  # List to collect calculated precision score for each classifier

# Iterating through key-value pairs in the clfs dictionary (model name -> model instance)
for name, clfs in clfs.items():  # Loops through all 11 classifier algorithms
    # Train current classifier and evaluate accuracy & precision on test set
    current_accuracy, current_precision = train_classifier(clfs, X_train, y_train, X_test, y_test)  # Calls train_classifier function
    
    # Print empty line break for readable console output
    print()
    # Print current classifier name
    print("For: ", name)  # Prints classifier name string
    # Print calculated test accuracy score
    print("Accuracy: ", current_accuracy)  # Prints accuracy value
    # Print calculated test precision score
    print("Precision: ", current_precision)  # Prints precision value
    
    # Append current accuracy score to accuracy_scores list
    accuracy_scores.append(current_accuracy)  # Appends score float
    # Append current precision score to precision_scores list
    precision_scores.append(current_precision)  # Appends score float



For:  SVC
Accuracy:  0.9661508704061895
Precision:  0.9327731092436975

For:  KNN
Accuracy:  0.9274661508704062
Precision:  1.0

For:  NB
Accuracy:  0.9709864603481625
Precision:  0.9655172413793104

For:  DT
Accuracy:  0.9381044487427466
Precision:  0.9021739130434783

For:  LR
Accuracy:  0.9632495164410058
Precision:  0.9629629629629629

For:  RF
Accuracy:  0.971953578336557
Precision:  0.943089430894309

For:  Adaboost
Accuracy:  0.9613152804642167
Precision:  0.9375

For:  Bgc
Accuracy:  0.965183752417795
Precision:  0.9180327868852459

For:  ETC
Accuracy:  0.9729206963249516
Precision:  0.9296875

For:  GBDT
Accuracy:  0.9506769825918762
Precision:  0.9393939393939394

For:  xgb
Accuracy:  0.9709864603481625
Precision:  0.9576271186440678
